# Спринт ревью №5

## Тема: Разработка информационной системы для учета заказов в компании

**Цель работы:** Создать программу для учета заказов в компании, используя классы, коллекции, делегаты и события.

## В ходе работы:

1. Созданы классы `Company`, `Employee`, `Customer` и `Order`.
2. Создан базовый класс `Person`, от которого наследуются классы `Employee` и `Customer`.
3. Реализованы принципы инкапсуляции, наследования и полиморфизма.
4. Созданы коллекции для хранения сотрудников, заказчиков и заказов.
5. Реализовано автоматическое присвоение уникального номера каждому заказу.
6. Создано перечисление `OrderStatus` для хранения статусов заказов.
7. Реализовано назначение ответственного сотрудника на заказ.
8. Созданы делегат и событие для уведомления об изменении статуса заказа.
9. Реализован поиск заказов по статусу и заказчику.
10. Реализован отчет по количеству обработанных заказов каждого сотрудника.
11. Проведено тестирование основных классов и методов.

In [2]:
using System;
using System.Collections.Generic;

// Перечисление статусов заказа
public enum OrderStatus
{
    Created,
    InProgress,
    Completed
}

// Делегат для уведомлений
public delegate void OrderNotification(Order order);

// Базовый класс человека
public class Person
{
    public string FirstName { get; private set; }
    public string LastName { get; private set; }

    public Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }

    public string GetFullName()
    {
        return FirstName + " " + LastName;
    }

    public void ChangeName(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }

    public virtual string GetInfo()
    {
        return GetFullName();
    }
}

// Класс сотрудника
public class Employee : Person
{
    public string Position { get; private set; }
    public int OrdersProcessed { get; private set; }

    public Employee(string firstName, string lastName, string position)
        : base(firstName, lastName)
    {
        Position = position;
        OrdersProcessed = 0;
    }

    public void ChangePosition(string position)
    {
        Position = position;
    }

    public void CompleteOrder()
    {
        OrdersProcessed++;
    }

    public override string GetInfo()
    {
        return $"{GetFullName()} — {Position}";
    }

    public void Notify(Order order)
    {
        Console.WriteLine($"Уведомление для {GetFullName()}: заказ №{order.OrderId} — {order.GetStatusName()}");
    }
}

// Класс заказчика
public class Customer : Person
{
    public string PhoneNumber { get; private set; }

    public Customer(string firstName, string lastName, string phoneNumber)
        : base(firstName, lastName)
    {
        PhoneNumber = phoneNumber;
    }

    public void ChangePhoneNumber(string phoneNumber)
    {
        PhoneNumber = phoneNumber;
    }

    public string GetPhoneNumber()
    {
        return PhoneNumber;
    }

    public override string GetInfo()
    {
        return $"{GetFullName()} — Телефон: {GetPhoneNumber()}";
    }
}

// Класс заказа
public class Order
{
    private static int nextId = 1;

    public int OrderId { get; private set; }
    public string Description { get; private set; }
    public Employee AssignedEmployee { get; private set; }
    public Customer Customer { get; private set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; private set; }

    public event OrderNotification StatusChanged;

    public Order(string description, Customer customer)
    {
        OrderId = nextId++;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void AssignEmployee(Employee employee)
    {
        AssignedEmployee = employee;
    }

    public void ChangeStatus(OrderStatus status)
    {
        if (Status == status || Status == OrderStatus.Completed)
        {
            return;
        }

        Status = status;
        StatusChanged?.Invoke(this);
    }

    public string GetStatusName()
    {
        if (Status == OrderStatus.Created)
        {
            return "Создан";
        }

        if (Status == OrderStatus.InProgress)
        {
            return "В работе";
        }

        return "Завершен";
    }

    public string GetInfo()
    {
        return $"Заказ №{OrderId}: {Description} — {GetStatusName()}";
    }
}

// Класс компании
public class Company
{
    public string Name { get; private set; }

    private List<Employee> employees = new List<Employee>();
    private List<Customer> customers = new List<Customer>();
    private List<Order> orders = new List<Order>();

    public Company(string name)
    {
        Name = name;
    }

    public void AddEmployee(Employee employee)
    {
        employees.Add(employee);
    }

    public void AddCustomer(Customer customer)
    {
        customers.Add(customer);
    }

    public Order AddOrder(string description, Customer customer)
    {
        Order order = new Order(description, customer);
        orders.Add(order);

        order.StatusChanged += NotifyEmployee;

        return order;
    }

    public void AssignEmployee(Order order, Employee employee)
    {
        if (orders.Contains(order) && employees.Contains(employee))
        {
            order.AssignEmployee(employee);
        }
    }

    private void NotifyEmployee(Order order)
    {
        if (order.AssignedEmployee != null)
        {
            order.AssignedEmployee.Notify(order);

            if (order.Status == OrderStatus.Completed)
            {
                order.AssignedEmployee.CompleteOrder();
            }
        }
    }

    public void ShowOrdersByStatus(OrderStatus status)
    {
        foreach (Order order in orders)
        {
            if (order.Status == status)
            {
                Console.WriteLine(order.GetInfo());
            }
        }
    }

    public void SearchOrdersByCustomer(Customer customer)
    {
        foreach (Order order in orders)
        {
            if (order.Customer == customer)
            {
                Console.WriteLine(order.GetInfo());
            }
        }
    }

    public void ShowReport()
    {
        foreach (Employee employee in employees)
        {
            Console.WriteLine($"{employee.GetInfo()} — Обработано заказов: {employee.OrdersProcessed}");
        }
    }
}

// Создание компании
Company company = new Company("ТехноСервис");

Console.WriteLine($"Компания: {company.Name}");
Console.WriteLine();

// Создание сотрудников
Employee employee1 = new Employee("Иван", "Иванов", "Менеджер");
Employee employee2 = new Employee("Анна", "Петрова", "Специалист");

company.AddEmployee(employee1);
company.AddEmployee(employee2);

// Создание заказчиков
Customer customer1 = new Customer("Мария", "Смирнова", "89000000001");
Customer customer2 = new Customer("Алексей", "Соколов", "89000000002");

company.AddCustomer(customer1);
company.AddCustomer(customer2);

// Создание заказов
Order order1 = company.AddOrder("Ремонт компьютера", customer1);
Order order2 = company.AddOrder("Установка программы", customer2);
Order order3 = company.AddOrder("Настройка принтера", customer1);

// Назначение ответственных сотрудников
company.AssignEmployee(order1, employee1);
company.AssignEmployee(order2, employee2);
company.AssignEmployee(order3, employee1);

// Изменение статусов заказов
Console.WriteLine("Уведомления:");

order1.ChangeStatus(OrderStatus.InProgress);
order1.ChangeStatus(OrderStatus.Completed);
order2.ChangeStatus(OrderStatus.InProgress);

Console.WriteLine();

// Просмотр заказов по статусам
Console.WriteLine("Созданные заказы:");
company.ShowOrdersByStatus(OrderStatus.Created);

Console.WriteLine();

Console.WriteLine("Заказы в работе:");
company.ShowOrdersByStatus(OrderStatus.InProgress);

Console.WriteLine();

Console.WriteLine("Завершенные заказы:");
company.ShowOrdersByStatus(OrderStatus.Completed);

Console.WriteLine();

// Поиск заказов по заказчику
Console.WriteLine($"Заказы клиента {customer1.GetFullName()}:");
company.SearchOrdersByCustomer(customer1);

Console.WriteLine();

// Проверка полиморфизма
Console.WriteLine("Информация о сотруднике и заказчике:");

Person person1 = employee1;
Person person2 = customer1;

Console.WriteLine(person1.GetInfo());
Console.WriteLine(person2.GetInfo());

Console.WriteLine();

// Отчет по сотрудникам
Console.WriteLine("Отчет по сотрудникам:");
company.ShowReport();

Компания: ТехноСервис

Уведомления:
Уведомление для Иван Иванов: заказ №1 — В работе
Уведомление для Иван Иванов: заказ №1 — Завершен
Уведомление для Анна Петрова: заказ №2 — В работе

Созданные заказы:
Заказ №3: Настройка принтера — Создан

Заказы в работе:
Заказ №2: Установка программы — В работе

Завершенные заказы:
Заказ №1: Ремонт компьютера — Завершен

Заказы клиента Мария Смирнова:
Заказ №1: Ремонт компьютера — Завершен
Заказ №3: Настройка принтера — Создан

Информация о сотруднике и заказчике:
Иван Иванов — Менеджер
Мария Смирнова — Телефон: 89000000001

Отчет по сотрудникам:
Иван Иванов — Менеджер — Обработано заказов: 1
Анна Петрова — Специалист — Обработано заказов: 0
